In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import sys
import json
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
src_path = os.getenv("SRC_PATH") 
data_path = os.getenv("DATA_PATH") 
transaction_path = os.path.join(data_path, r'raw/train_transaction.csv/train_transaction.csv')
identity_path = os.path.join(data_path, r'raw/train_identity.csv/train_identity.csv')
train_transaction = pd.read_csv(transaction_path)
train_identity = pd.read_csv(identity_path)
full_df = train_transaction.merge(train_identity, on="TransactionID", how = "left")
full_df = full_df.sort_values("TransactionDT").reset_index(drop=True)

In [4]:
sys.path.append(src_path)
from features.engineering import create_engineered_features
from data.split import temporal_split

In [5]:
json_path = os.path.join(data_path, "processed", "split_info.json")
with open(json_path, 'r') as f:
    split_info = json.load(f)

train_end = split_info.get("train_end")
val_end = split_info.get("validation_end")

In [6]:
train_df, val_df, test_df = temporal_split(full_df, train_end, val_end)
y_datasets = {}
map_dfs = {"train": train_df, "val": val_df, "test": test_df}

for name, subset_df in map_dfs.items():
    y_datasets[f'y_{name}'] = subset_df['isFraud']

In [7]:
def add_amount_features(df):
    df = df.copy()

    df["transaction_amt_log"] = np.log1p(df["TransactionAmt"])

    df["amount_decimal"] = (df["TransactionAmt"] % 1)

    return df

In [8]:
def create_d_time_features(df):
    df = df.copy()

    df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)

    df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)

    return df

In [ ]:
def add_email_features(df):
    df = df.copy()

    unsusual_emails = d_features_dfs["train"]["R_emaildomain"].value_counts().head(10).index.tolist()
    
    for col in ["P_emaildomain", "R_emaildomain"]:
        df[f'{col}_is_missing'] = df[col].isna().astype(int)

        df[f"{col}_provider"] = df[col].fillna("missing").str.split(".").str[0]

    df['domain_math'] = (df["P_emaildomain"].fillna("missing") == df["R_emaildomain"].fillna("missing")).astype(int)

    df['is_anusual_email'] = (df["P_emaildomain"] in unsusual_emails | df["R_emaildomain"] in unsusual_emails).astype(int)

    return df

In [ ]:
def add_combined_features(df):
    df = df.copy()
    df["card_product"] = df["card4"].astype(str) + "_" + df["ProductCD"].astype(str)
    df["email_product"] = df["P_emaildomain"].astype(str) + "_" + df["ProductCD"].astype(str)
    df["card_addr"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str)
    return df

In [ ]:
def add_additional_features(df):
    df = df.copy()
    df["dist_discrepancy"] = (df["dist1"] - df["dist2"]).abs()
    return df

In [ ]:
def create_d_features(df):
    df = create_engineered_features(df)
    df = add_amount_features(df)
    df = create_d_time_features(df)
    df = add_email_features(df)
    df = add_combined_features(df)
    df = add_additional_features(df)
    return df

In [11]:
d_features_dfs = {}

d_features_dfs.clear()
for name, subset_df in map_dfs.items():
    d_features_dfs[f'{name}'] = create_d_features(subset_df)

In [12]:
assert list(d_features_dfs['train'].columns) == list(d_features_dfs['val'].columns)
assert list(d_features_dfs['train']) == list(d_features_dfs['test'].columns)

In [13]:
def get_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()

    num_cols = X.select_dtypes(include=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))])


    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)])

    return preprocessor

def create_pipeline(X):
    preprocessor = get_preprocessor(X)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear"))])

    return pipe

def evaluate_model(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:,1]

    return {"pr_auc": average_precision_score(y, probabilities),
            "roc_auc": roc_auc_score(y, probabilities),
            "precision": precision_score(y, predictions, zero_division=0),
            "recall": recall_score(y, predictions, zero_division=0),
            "f1": f1_score(y, predictions, zero_division=0)
            }

In [14]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name="fraud-detection-baseline")
result = []
with mlflow.start_run(run_name="Logistic_regression_D_features"):
    X_train = d_features_dfs['train']
    pipe = create_pipeline(X_train)

    pipe.fit(X_train, y_datasets['y_train'])

    metrics = evaluate_model(pipe, d_features_dfs['val'], y_datasets['y_val'])

    mlflow.log_param("model", "logistic_regression")

    mlflow.log_param("class_weight", "balanced")

    mlflow.log_param("feature_count", X_train.shape[1])

    mlflow.log_metrics(metrics)

    result.append({**metrics})

🏃 View run Logistic_regression_D_features at: http://127.0.0.1:5000/#/experiments/1/runs/435464eea07d41c7a4d2c04c0c7e81b9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [15]:
print("X_train shape:", d_features_dfs["train"].shape)
print("X_val shape:", d_features_dfs["val"].shape)

X_train shape: (413378, 29)
X_val shape: (88581, 29)


In [ ]:
assert len(d_features_dfs["train"]) == len(y_datasets["y_train"]), "Row count mismatch!"
assert "isFraud" not in d_features_dfs["train"].columns, "Target leakage detected!"

In [19]:
d_features_dfs["train"]

,TransactionDT,TransactionAmt,ProductCD,P_emaildomain,R_emaildomain,card2,card4,card6,addr1,addr2,dist1,dist2,missing_count,missing_V_count,missing_D_count,missing_M_count,missing_C_count,missing_id_count,missing_card_count,transaction_day,transaction_hour,transaction_amt_log,amount_decimal,hour_sin,hour_cos,P_emaildomain_is_missing,P_emaildomain_provider,R_emaildomain_is_missing,R_emaildomain_provider
0,86400,68.50,W,NaN,NaN,NaN,discover,credit,315.0,87.0,19.0,NaN,4,0.0,0.0,0.0,0.0,0.0,1,1,0,4.241327,0.50,0.000000,1.000000,1,missing,1,missing
1,86401,29.00,W,gmail.com,NaN,404.0,mastercard,credit,325.0,87.0,NaN,NaN,3,0.0,0.0,0.0,0.0,0.0,0,1,0,3.401197,0.00,0.000000,1.000000,0,gmail,1,missing
2,86469,59.00,W,outlook.com,NaN,490.0,visa,debit,330.0,87.0,287.0,NaN,2,0.0,0.0,0.0,0.0,0.0,0,1,0,4.094345,0.00,0.000000,1.000000,0,outlook,1,missing
3,86499,50.00,W,yahoo.com,NaN,567.0,mastercard,debit,476.0,87.0,NaN,NaN,3,0.0,0.0,0.0,0.0,0.0,0,1,0,3.931826,0.00,0.000000,1.000000,0,yahoo,1,missing
4,86506,50.00,H,gmail.com,NaN,514.0,mastercard,credit,420.0,87.0,NaN,NaN,3,0.0,0.0,0.0,0.0,0.0,0,1,0,3.931826,0.00,0.000000,1.000000,0,gmail,1,missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413373,10437970,47.95,W,hotmail.com,NaN,111.0,visa,debit,126.0,87.0,NaN,NaN,3,0.0,0.0,0.0,0.0,0.0,0,120,19,3.890799,0.95,-0.965926,0.258819,0,hotmail,1,missing
413374,10437974,226.00,W,hotmail.com,NaN,360.0,visa,debit,512.0,87.0,3.0,NaN,2,0.0,0.0,0.0,0.0,0.0,0,120,19,5.424950,0.00,-0.965926,0.258819,0,hotmail,1,missing
413375,10437990,72.00,W,gmail.com,NaN,271.0,visa,debit,330.0,87.0,NaN,NaN,3,0.0,0.0,0.0,0.0,0.0,0,120,19,4.290459,0.00,-0.965926,0.258819,0,gmail,1,missing
413376,10437992,35.00,W,gmail.com,NaN,170.0,mastercard,debit,325.0,87.0,NaN,NaN,3,0.0,0.0,0.0,0.0,0.0,0,120,19,3.583519,0.00,-0.965926,0.258819,0,gmail,1,missing


In [29]:
unsusual_emails = d_features_dfs["train"]["R_emaildomain"].value_counts().head(20).index.tolist()